# evaluate 4d flow inference :


#### - The network : `nnUnet`
#### - Data trained on: `RACELLETE`
#### - Inferenced images: `enhanced magnitude image from AHC's data`

## get GT from standard mri ---> copy label to gt_d4flow folder

ground truth orientation not the same as inferred labels 

In [4]:
import pandas as pd
from pathlib import Path
import os
import SimpleITK as sitk

# =====================================================
# Paths
# =====================================================

BASE_DIR = os.path.dirname(os.path.abspath(os.getcwd()))

csv_file = os.path.join(
    BASE_DIR,
    "workspace/mrns_dictionary.csv"
)

folder1 = Path(
    os.path.join(
        BASE_DIR,
        "workspace/sample_data/ao_segs"
    )
)

folder2 = Path(
    os.path.join(
        BASE_DIR,
        "workspace/train_data/ao_segs"
    )
)

destination = Path(
    os.path.join(
        BASE_DIR,
        "workspace/gt_4dflows_pir"
    )
)

destination.mkdir(parents=True, exist_ok=True)

TARGET_ORIENTATION = "PIR"


# =====================================================
# Helpers
# =====================================================

def get_orientation(img):
    return sitk.DICOMOrientImageFilter_GetOrientationFromDirectionCosines(
        img.GetDirection()
    )


# =====================================================
# Read MRNs
# =====================================================

df = pd.read_csv(csv_file, dtype={"mrn": str})
mrns = set(df["mrn"].dropna().astype(str))


# =====================================================
# Copy + Reorient
# =====================================================

for mrn in sorted(mrns):

    found = False

    for folder in [folder1, folder2]:

        matches = list(folder.glob(f"*{mrn}*.nii.gz"))

        if matches:

            for src_file in matches:

                try:
                    img = sitk.ReadImage(str(src_file))

                    before_orientation = get_orientation(img)
                    before_size = img.GetSize()

                    # Reorient to PIR
                    img_pir = sitk.DICOMOrient(
                        img,
                        TARGET_ORIENTATION
                    )

                    after_orientation = get_orientation(img_pir)
                    after_size = img_pir.GetSize()

                    dst_file = destination / src_file.name

                    sitk.WriteImage(
                        img_pir,
                        str(dst_file)
                    )

                    print(
                        f"\nProcessed: {src_file.name}"
                    )
                    print(
                        f"  Before: {before_orientation} "
                        f"{before_size}"
                    )
                    print(
                        f"  After : {after_orientation} "
                        f"{after_size}"
                    )
                    print(
                        f"  Saved : {dst_file}"
                    )

                except Exception as e:
                    print(
                        f"Error processing "
                        f"{src_file.name}: {e}"
                    )

            found = True
            break

    if not found:
        print(f"Not found: {mrn}")

print("\nFinished reorienting all available GT files.")


Processed: AORTA_29149.nii.gz
  Before: RAS (60, 156, 240)
  After : PIR (156, 240, 60)
  Saved : /workspace/gt_4dflows_pir/AORTA_29149.nii.gz

Processed: AORTA_29241.nii.gz
  Before: RAS (72, 156, 240)
  After : PIR (156, 240, 72)
  Saved : /workspace/gt_4dflows_pir/AORTA_29241.nii.gz

Processed: AORTA_29255.nii.gz
  Before: RAS (72, 156, 240)
  After : PIR (156, 240, 72)
  Saved : /workspace/gt_4dflows_pir/AORTA_29255.nii.gz

Processed: AORTA_29336.nii.gz
  Before: RAS (64, 156, 240)
  After : PIR (156, 240, 64)
  Saved : /workspace/gt_4dflows_pir/AORTA_29336.nii.gz

Processed: AORTA_29465.nii.gz
  Before: RAS (64, 156, 240)
  After : PIR (156, 240, 64)
  Saved : /workspace/gt_4dflows_pir/AORTA_29465.nii.gz
Not found: 29528

Processed: AORTA_30266.nii.gz
  Before: RAS (56, 156, 240)
  After : PIR (156, 240, 56)
  Saved : /workspace/gt_4dflows_pir/AORTA_30266.nii.gz

Processed: AORTA_31122.nii.gz
  Before: RAS (80, 156, 240)
  After : PIR (156, 240, 80)
  Saved : /workspace/gt_4dflow

## Evaluate enhanced labels before Connected Components:

In [6]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy.spatial.distance import cdist

In [ ]:
# import SimpleITK as sitk



# # 1. Load volumes
# fixed_image = sitk.ReadImage("sequence_with_seg.nii.gz")       # Shape: 280, 160, 22
# moving_image = sitk.ReadImage("sequence_to_predict.nii.gz")     # Shape: 206, 180, 20
# segmentation = sitk.ReadImage("ground_truth_seg.nii.gz")        # Shape: 280, 160, 22


# # 2. Setup registration framework
# registration_method = sitk.ImageRegistrationMethod()
# # Similarity metric: Mutual Information works best across different MRI sequences
# registration_method.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
# registration_method.SetMetricSamplingStrategy(registration_method.RANDOM)
# registration_method.SetMetricSamplingPercentage(0.15)
# # Interpolator for intensity evaluation during registration
# registration_method.SetInterpolator(sitk.sitkLinear)
# # Optimizer
# registration_method.SetOptimizerAsGradientDescent(learningRate=1.0, numberOfIterations=100)
# registration_method.SetOptimizerScalesFromPhysicalShift()
# # Initialize transform (Align centers of mass first)
# initial_transform = sitk.CenteredTransformInitializer(
#    fixed_image,
#    moving_image,
#    sitk.Euler3DTransform(),
#    sitk.CenteredTransformInitializerFilter.GEOMETRY
# )
# registration_method.SetInitialTransform(initial_transform, inPlace=False)
# # Execute registration
# final_transform = registration_method.Execute(fixed_image, moving_image)
# # 3. Resample Moving Image to Fixed Image's physical grid
# resampled_moving = sitk.Resample(
#    moving_image,
#    fixed_image,               # Use fixed image for size, spacing, origin, direction
#    final_transform,
#    sitk.sitkLinear,           # Linear interpolation for MRI intensities
#    0.0,                       # Default pixel value for out-of-bounds
#    moving_image.GetPixelID()
# )

In [10]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk

from scipy.spatial.distance import cdist


# =====================================================
# Paths
# =====================================================

BASE_DIR = os.path.dirname(os.path.abspath(os.getcwd()))

gt_dir = Path(
    os.path.join(
        BASE_DIR,
        "workspace/gt_4dflows_pir"
    )
)

pred_dir = Path(
    os.path.join(
        BASE_DIR,
        "workspace/nnUNet_raw/Dataset301_RACLETTE_4DFLOW_AORTA/labelsTs"
    )
)

mapping_csv = os.path.join(
    BASE_DIR,
    "workspace/mrns_dictionary.csv"
)


# =====================================================
# Helpers
# =====================================================

def print_info(path):

    img = sitk.ReadImage(str(path))

    orient = (
        sitk.DICOMOrientImageFilter_GetOrientationFromDirectionCosines(
            img.GetDirection()
        )
    )

    print(f"\n{path.name}")
    print("Size      :", img.GetSize())
    print("Spacing   :", img.GetSpacing())
    print("Origin    :", img.GetOrigin())
    print("Direction :", img.GetDirection())
    print("Orientation:", orient)


def resample_prediction_to_gt(
    pred_img,
    gt_img
):
    """
    Resample prediction label into GT geometry.
    """

    return sitk.Resample(
        pred_img,
        gt_img,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        pred_img.GetPixelID()
    )


# =====================================================
# Metrics
# =====================================================

def dice_score(gt, pred):

    gt = gt > 0
    pred = pred > 0

    intersection = np.logical_and(
        gt,
        pred
    ).sum()

    denom = gt.sum() + pred.sum()

    if denom == 0:
        return 1.0

    return 2.0 * intersection / denom


def iou_score(gt, pred):

    gt = gt > 0
    pred = pred > 0

    intersection = np.logical_and(
        gt,
        pred
    ).sum()

    union = np.logical_or(
        gt,
        pred
    ).sum()

    if union == 0:
        return 1.0

    return intersection / union


def volume_difference(gt, pred):

    gt_vol = np.sum(gt > 0)
    pred_vol = np.sum(pred > 0)

    return pred_vol - gt_vol


def hd95_mm(gt_array, pred_array, spacing):
    """
    HD95 in millimeters.

    spacing should be:
    (z_spacing, y_spacing, x_spacing)
    to match numpy array indexing.
    """

    gt_pts = np.argwhere(gt_array > 0)
    pred_pts = np.argwhere(pred_array > 0)

    if len(gt_pts) == 0 or len(pred_pts) == 0:
        return np.nan

    gt_pts = gt_pts.astype(np.float32)
    pred_pts = pred_pts.astype(np.float32)

    gt_pts *= np.array(spacing)
    pred_pts *= np.array(spacing)

    d_gt_to_pred = cdist(
        gt_pts,
        pred_pts
    ).min(axis=1)

    d_pred_to_gt = cdist(
        pred_pts,
        gt_pts
    ).min(axis=1)

    all_distances = np.concatenate(
        [
            d_gt_to_pred,
            d_pred_to_gt
        ]
    )

    return np.percentile(
        all_distances,
        95
    )


# =====================================================
# Load mapping
# =====================================================

mapping_df = pd.read_csv(
    mapping_csv,
    dtype=str
)

results = []


# =====================================================
# Evaluation loop
# =====================================================

for _, row in mapping_df.iterrows():

    mrn = row["mrn"].strip()
    vsid = row["id"].strip()

    gt_file = gt_dir / f"AORTA_{mrn}.nii.gz"

    if not gt_file.exists():

        print(
            f"Missing GT: {gt_file.name}"
        )
        continue

    pred_files = sorted(
        pred_dir.glob(
            f"{vsid}_f*.nii.gz"
        )
    )

    if len(pred_files) == 0:

        print(
            f"No prediction volumes found for {vsid}"
        )
        continue

    print_info(gt_file)
    print_info(pred_files[0])

    print(
        f"\nEvaluating MRN={mrn} "
        f"({len(pred_files)} volumes)"
    )

    gt_img = sitk.ReadImage(
        str(gt_file)
    )

    gt = sitk.GetArrayFromImage(
        gt_img
    )

    # Convert SITK spacing (x,y,z)
    # to NumPy order (z,y,x)
    spacing_zyx = (
        gt_img.GetSpacing()[2],
        gt_img.GetSpacing()[1],
        gt_img.GetSpacing()[0]
    )

    for pred_file in pred_files:

        pred_img = sitk.ReadImage(
            str(pred_file)
        )

        # ---------------------------------
        # Resample prediction to GT geometry
        # ---------------------------------

        pred_resampled_img = (
            resample_prediction_to_gt(
                pred_img,
                gt_img
            )
        )

        pred = sitk.GetArrayFromImage(
            pred_resampled_img
        )

        if gt.shape != pred.shape:

            print(
                f"Unexpected mismatch "
                f"{gt.shape} vs {pred.shape}"
            )
            continue

        dice = dice_score(
            gt,
            pred
        )

        iou = iou_score(
            gt,
            pred
        )

        hd = hd95_mm(
            gt,
            pred,
            spacing_zyx
        )

        vol_diff = volume_difference(
            gt,
            pred
        )

        results.append(
            {
                "mrn": mrn,
                "vsid": vsid,
                "volume_file": pred_file.name,
                "dice": dice,
                "iou": iou,
                "hd95_mm": hd,
                "volume_difference": vol_diff,
            }
        )

        print(
            f"{pred_file.name} | "
            f"Dice={dice:.4f} | "
            f"IoU={iou:.4f} | "
            f"HD95(mm)={hd:.2f}"
        )


# =====================================================
# Save results
# =====================================================

results_df = pd.DataFrame(results)

results_csv = (
    "/workspace/enhanced_4dflows_evaluation_results.csv"
)

results_df.to_csv(
    results_csv,
    index=False
)

print("\nResults saved:")
print(results_csv)


# =====================================================
# Best volume per MRN
# =====================================================

if len(results_df) > 0:

    best_df = (
        results_df
        .sort_values(
            "dice",
            ascending=False
        )
        .groupby("mrn")
        .first()
        .reset_index()
    )

    best_csv = (
        "/workspace/best_volume_per_patient.csv"
    )

    best_df.to_csv(
        best_csv,
        index=False
    )

    print(
        "\nBest volume per patient saved:"
    )
    print(best_csv)


# =====================================================
# Summary
# =====================================================

if len(results_df) > 0:

    print("\nOverall Summary")

    print(
        results_df[
            [
                "dice",
                "iou",
                "hd95_mm"
            ]
        ].describe()
    )


AORTA_29149.nii.gz
Size      : (156, 240, 60)
Spacing   : (2.0833332538604736, 2.0833330154418945, 1.600018858909607)
Origin    : (15.520461082458496, -131.7872314453125, 243.27825927734375)
Direction : (0.3080029866399324, -0.058684917239886954, -0.9495737097007431, 0.9512130815971336, 1.6785392616434862e-07, 0.30853473606710724, -0.018106176276799515, -0.9982765551131206, 0.05582191760562723)
Orientation: PIR

VS001_f00.nii.gz
Size      : (126, 208, 24)
Spacing   : (1.9230769872665405, 1.9230769872665405, 2.0)
Origin    : (-2.8015191555023193, -101.27269744873047, 200.4728546142578)
Direction : (0.30745202960207796, -0.023703288671894095, -0.9512683140467567, 0.9515635814245745, 0.007658564443770314, 0.30735662538551284, -1.2964922628897435e-08, -0.9996897021060073, 0.024909827824993045)
Orientation: PIR

Evaluating MRN=29149 (22 volumes)
VS001_f00.nii.gz | Dice=0.6456 | IoU=0.4766 | HD95(mm)=39.58
VS001_f01.nii.gz | Dice=0.6590 | IoU=0.4914 | HD95(mm)=36.92
VS001_f02.nii.gz | Dice=

## evaluating segmentation after computing connected components

In [11]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk

from scipy.spatial.distance import cdist


# =====================================================
# Paths
# =====================================================

BASE_DIR = os.path.dirname(os.path.abspath(os.getcwd()))

gt_dir = Path(
    os.path.join(
        BASE_DIR,
        "workspace/gt_4dflows_pir"
    )
)

pred_dir = Path(
    os.path.join(
        BASE_DIR,
        "workspace/4dflows_magnitude_ahc/labelsTs_enhanced_lcc"
    )
)

mapping_csv = os.path.join(
    BASE_DIR,
    "workspace/mrns_dictionary.csv"
)


# =====================================================
# Helpers
# =====================================================

def print_info(path):

    img = sitk.ReadImage(str(path))

    orient = (
        sitk.DICOMOrientImageFilter_GetOrientationFromDirectionCosines(
            img.GetDirection()
        )
    )

    print(f"\n{path.name}")
    print("Size      :", img.GetSize())
    print("Spacing   :", img.GetSpacing())
    print("Origin    :", img.GetOrigin())
    print("Direction :", img.GetDirection())
    print("Orientation:", orient)


def resample_prediction_to_gt(
    pred_img,
    gt_img
):
    """
    Resample prediction label into GT geometry.
    """

    return sitk.Resample(
        pred_img,
        gt_img,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        pred_img.GetPixelID()
    )


# =====================================================
# Metrics
# =====================================================

def dice_score(gt, pred):

    gt = gt > 0
    pred = pred > 0

    intersection = np.logical_and(
        gt,
        pred
    ).sum()

    denom = gt.sum() + pred.sum()

    if denom == 0:
        return 1.0

    return 2.0 * intersection / denom


def iou_score(gt, pred):

    gt = gt > 0
    pred = pred > 0

    intersection = np.logical_and(
        gt,
        pred
    ).sum()

    union = np.logical_or(
        gt,
        pred
    ).sum()

    if union == 0:
        return 1.0

    return intersection / union


def volume_difference(gt, pred):

    gt_vol = np.sum(gt > 0)
    pred_vol = np.sum(pred > 0)

    return pred_vol - gt_vol


def hd95_mm(gt_array, pred_array, spacing):
    """
    HD95 in millimeters.

    spacing should be:
    (z_spacing, y_spacing, x_spacing)
    to match numpy array indexing.
    """

    gt_pts = np.argwhere(gt_array > 0)
    pred_pts = np.argwhere(pred_array > 0)

    if len(gt_pts) == 0 or len(pred_pts) == 0:
        return np.nan

    gt_pts = gt_pts.astype(np.float32)
    pred_pts = pred_pts.astype(np.float32)

    gt_pts *= np.array(spacing)
    pred_pts *= np.array(spacing)

    d_gt_to_pred = cdist(
        gt_pts,
        pred_pts
    ).min(axis=1)

    d_pred_to_gt = cdist(
        pred_pts,
        gt_pts
    ).min(axis=1)

    all_distances = np.concatenate(
        [
            d_gt_to_pred,
            d_pred_to_gt
        ]
    )

    return np.percentile(
        all_distances,
        95
    )


# =====================================================
# Load mapping
# =====================================================

mapping_df = pd.read_csv(
    mapping_csv,
    dtype=str
)

results = []


# =====================================================
# Evaluation loop
# =====================================================

for _, row in mapping_df.iterrows():

    mrn = row["mrn"].strip()
    vsid = row["id"].strip()

    gt_file = gt_dir / f"AORTA_{mrn}.nii.gz"

    if not gt_file.exists():

        print(
            f"Missing GT: {gt_file.name}"
        )
        continue

    pred_files = sorted(
        pred_dir.glob(
            f"{vsid}_f*.nii.gz"
        )
    )

    if len(pred_files) == 0:

        print(
            f"No prediction volumes found for {vsid}"
        )
        continue

    print_info(gt_file)
    print_info(pred_files[0])

    print(
        f"\nEvaluating MRN={mrn} "
        f"({len(pred_files)} volumes)"
    )

    gt_img = sitk.ReadImage(
        str(gt_file)
    )

    gt = sitk.GetArrayFromImage(
        gt_img
    )

    # Convert SITK spacing (x,y,z)
    # to NumPy order (z,y,x)
    spacing_zyx = (
        gt_img.GetSpacing()[2],
        gt_img.GetSpacing()[1],
        gt_img.GetSpacing()[0]
    )

    for pred_file in pred_files:

        pred_img = sitk.ReadImage(
            str(pred_file)
        )

        # ---------------------------------
        # Resample prediction to GT geometry
        # ---------------------------------

        pred_resampled_img = (
            resample_prediction_to_gt(
                pred_img,
                gt_img
            )
        )

        pred = sitk.GetArrayFromImage(
            pred_resampled_img
        )

        if gt.shape != pred.shape:

            print(
                f"Unexpected mismatch "
                f"{gt.shape} vs {pred.shape}"
            )
            continue

        dice = dice_score(
            gt,
            pred
        )

        iou = iou_score(
            gt,
            pred
        )

        hd = hd95_mm(
            gt,
            pred,
            spacing_zyx
        )

        vol_diff = volume_difference(
            gt,
            pred
        )

        results.append(
            {
                "mrn": mrn,
                "vsid": vsid,
                "volume_file": pred_file.name,
                "dice": dice,
                "iou": iou,
                "hd95_mm": hd,
                "volume_difference": vol_diff,
            }
        )

        print(
            f"{pred_file.name} | "
            f"Dice={dice:.4f} | "
            f"IoU={iou:.4f} | "
            f"HD95(mm)={hd:.2f}"
        )


# =====================================================
# Save results
# =====================================================

results_df = pd.DataFrame(results)

results_csv = (
    "/workspace/enhanced_4dflows_lcc_evaluation_results.csv"
)

results_df.to_csv(
    results_csv,
    index=False
)

print("\nResults saved:")
print(results_csv)


# =====================================================
# Best volume per MRN
# =====================================================

if len(results_df) > 0:

    best_df = (
        results_df
        .sort_values(
            "dice",
            ascending=False
        )
        .groupby("mrn")
        .first()
        .reset_index()
    )

    best_csv = (
        "/workspace/best_volume_per_patient.csv"
    )

    best_df.to_csv(
        best_csv,
        index=False
    )

    print(
        "\nBest volume per patient saved:"
    )
    print(best_csv)


# =====================================================
# Summary
# =====================================================

if len(results_df) > 0:

    print("\nOverall Summary")

    print(
        results_df[
            [
                "dice",
                "iou",
                "hd95_mm"
            ]
        ].describe()
    )


AORTA_29149.nii.gz
Size      : (156, 240, 60)
Spacing   : (2.0833332538604736, 2.0833330154418945, 1.600018858909607)
Origin    : (15.520461082458496, -131.7872314453125, 243.27825927734375)
Direction : (0.3080029866399324, -0.058684917239886954, -0.9495737097007431, 0.9512130815971336, 1.6785392616434862e-07, 0.30853473606710724, -0.018106176276799515, -0.9982765551131206, 0.05582191760562723)
Orientation: PIR

VS001_f00.nii.gz
Size      : (126, 208, 24)
Spacing   : (1.9230769872665405, 1.9230769872665405, 2.0)
Origin    : (-2.8015191555023193, -101.27269744873047, 200.4728546142578)
Direction : (0.30745202960207796, -0.023703288671894095, -0.9512683140467567, 0.9515635814245745, 0.007658564443770314, 0.30735662538551284, -1.2968543554475485e-08, -0.9996897021060073, 0.024909827824993045)
Orientation: PIR

Evaluating MRN=29149 (22 volumes)
VS001_f00.nii.gz | Dice=0.6458 | IoU=0.4768 | HD95(mm)=39.21
VS001_f01.nii.gz | Dice=0.6590 | IoU=0.4914 | HD95(mm)=36.92
VS001_f02.nii.gz | Dice=